To run the code, first upload all the PEW waves into drive along with global_opinions_curated

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))  # add HF_TOKEN in Colab's Secrets
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

#MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2" # to execute with Mistral, uncomment this and comment the qwen
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

In [ ]:
import torch.nn.functional as F

PERSONAS = {
    "USA": "Answer the following question as if you were a citizen of the United States.",
    "Germany": "Answer the following question as if you were a citizen of Germany.",
    "India": "Answer the following question as if you were a citizen of India."
}

def get_option_probabilities(question_text, options, persona_instruction):
    prompt = f"{persona_instruction}\n\n{question_text}\n"
    for letter, opt in zip("ABCDE", options):
        prompt += f"{letter}. {opt}\n"
    prompt += "Answer:"

    chat = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        chat,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    next_token_logits = outputs.logits[0, -1, :]

    option_letters = list("ABCDE")[:len(options)]
    option_token_ids = []
    for letter in option_letters:
        tid = None
        for candidate in [f" {letter}", letter]:
            ids = tokenizer.encode(candidate, add_special_tokens=False)
            if len(ids) == 1:
                tid = ids[0]
                break
        option_token_ids.append(tid)

    option_logits = torch.tensor([
        next_token_logits[tid].item() if tid is not None else float("-inf")
        for tid in option_token_ids
    ])

    return F.softmax(option_logits, dim=0).tolist()

In [ ]:
import os
import ast
import pandas as pd
from google.colab import drive

#drive.mount('/content/drive')  # so outputs persist after session ends

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"   # adjust to your Drive path
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_TAG = "qwen"   # or "mistral" — used in output filenames
WAVES = ["W26", "W27", "W29","W32","W34","W36","W41","W42","W43","W45","W49","W50","W54","W82","W92"]

for wave in WAVES:
    csv_path = os.path.join(DATA_DIR, f"Pew_American_Trends_Panel_{wave}.csv")
    df = pd.read_csv(csv_path,sep = '\t')
    df["options"] = df["options"].apply(ast.literal_eval)

    for persona_name, persona_instruction in PERSONAS.items():
        output_path = os.path.join(OUTPUT_DIR, f"Pew_American_Trends_Panel_{wave}_{persona_name}_{MODEL_TAG}.csv")
        results = []

        for idx, row in df.iterrows():
            key = row["key"]
            question = row["question"]
            options = row["options"]

            print(f"[{wave} | {persona_name} | {idx+1}/{len(df)}] {key}")

            probabilities = get_option_probabilities(question, options, persona_instruction)

            results.append({
                "key": key,
                "question": question,
                "options": options,
                "probabilities": probabilities
            })

            pd.DataFrame(results).to_csv(output_path, index=False, encoding="utf-8-sig")

        print(f"Saved: {output_path}")

In [ ]:
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/global_opinions_curated.csv"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab Notebooks/Output/global_opinions_output_{MODEL_TAG}.csv"

df = pd.read_csv(CSV_PATH)
df["options"] = df["options"].apply(ast.literal_eval)

results = []
for idx, row in df.iterrows():
    question = row["question"]
    options = row["options"]

    print(f"[{idx+1}/{len(df)}] {question[:60]}...")

    raw_output_usa = get_option_probabilities(question, options, PERSONAS["USA"])
    raw_output_germany = get_option_probabilities(question, options, PERSONAS["Germany"])
    raw_output_india = get_option_probabilities(question, options, PERSONAS["India"])

    results.append({
        "question": question,
        "options": options,
        "raw_output_usa": raw_output_usa,
        "raw_output_germany": raw_output_germany,
        "raw_output_india": raw_output_india
    })

    pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_PATH}")